# ARM-H20 — Encoding, Apriori và Validation

Ngưỡng rời rạc hóa được học trên Train. Apriori sinh luật ứng viên trên Train; Validation đánh giá độ ổn định của luật trước lựa chọn cấu hình chung cho bốn horizon.

## 1. Cấu hình và dữ liệu

In [1]:
from pathlib import Path
from itertools import combinations, product
from datetime import datetime, timezone
import argparse
import hashlib
import json
import math

import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'data/modeling/horizons').is_dir() and (p/'notebook').is_dir())
OUT = ROOT / 'results/arm'
DATA_OUT = ROOT / 'data/arm'
HORIZONS = (1, 5, 10, 20)
FEATURES = [
    'Gold_Daily_Return', 'Gold_Return_Lag_1', 'Gold_Intraday_Range',
    'Gold_Volatility_20', 'Gold_Distance_to_MA_20', 'Gold_RSI_14',
    'Gold_MACD_Histogram', 'Oil_Price_Change', 'Dollar_Daily_Return',
    'SP500_Daily_Return', 'SP500_Drawdown',
]
QUANTILE_FEATURES = ['Gold_Intraday_Range', 'Gold_Volatility_20',
                     'Gold_RSI_14', 'SP500_Drawdown']
GRID = list(product((0.02, 0.05, 0.10), (0.50, 0.55, 0.60), (1.00, 1.05, 1.10)))
TARGET = 'Gold_Target_Next'
MIN_VAL_COUNT = 30


from IPython.display import display

H = 20
import pickle


In [2]:
def digest(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def folder(h):
    path = OUT / '1_train'
    path.mkdir(parents=True, exist_ok=True)
    return path

def load_data(h):
    source = ROOT / f'data/modeling/horizons/h{h}'
    data = pd.read_csv(source / 'modeling_dataset.csv', parse_dates=['Date'])
    trace = pd.read_csv(source / 'target_trace.csv', parse_dates=['Date', 'Target_Date'])
    data = data.merge(trace[['Date', 'Target_Date', 'Close', 'Target_Close']],
                      on='Date', validate='one_to_one')
    if data[FEATURES + [TARGET]].isna().any().any() or data.Date.duplicated().any():
        raise ValueError('Missing feature/target or duplicate date')
    if not data.Date.is_monotonic_increasing or not data.Target_Date.gt(data.Date).all():
        raise ValueError('Invalid date order')
    if not data[TARGET].eq((data.Target_Close > data.Close).astype(int)).all():
        raise ValueError('Target disagrees with price trace')
    return data

def split_masks(data):
    
    return {
        'train': (data.Date < '2021-01-01') & (data.Target_Date < '2021-01-01'),
        'val': data.Date.between('2021-01-01', '2023-12-31') & (data.Target_Date < '2024-01-01'),
        'test': data.Date >= '2024-01-01',
    }

## 2. Encoding từ Train
7 biến theo dấu; Range/Vol20/RSI theo quantile 1/3–2/3; Drawdown Deep/Medium/Shallow. Ngưỡng fit riêng trên Train đã purge. Không scaler, không window.

In [3]:
def fit_thresholds(train):
    rows = []
    for name in QUANTILE_FEATURES:
        q1, q2 = train[name].quantile([1/3, 2/3], interpolation='linear')
        rows.append({'Feature': name, 'q1': float(q1), 'q2': float(q2),
                     'Train_Rows': len(train), 'Interpolation': 'linear'})
    return pd.DataFrame(rows)

def encode(frame, thresholds, h):
    """Encode original unscaled values; quantile edges are fitted on Train only."""
    encoded = frame[['Date', 'Target_Date']].copy()
    cuts = thresholds.set_index('Feature')
    for name in FEATURES:
        values = frame[name]
        if name in QUANTILE_FEATURES:
            states = ['Deep', 'Medium', 'Shallow'] if name == 'SP500_Drawdown' else ['Low', 'Medium', 'High']
            q1, q2 = cuts.loc[name, ['q1', 'q2']]
            state = np.select([values <= q1, values <= q2], states[:2], default=states[2])
        else:
            
            state = np.select([values < 0, values == 0], ['Negative', 'Zero'], default='Positive')
        encoded[name] = name + '=' + pd.Series(state, index=frame.index)
    encoded['Target_Item'] = np.where(frame[TARGET] == 1, f'Gold_Up_H{h}', f'Gold_NonUp_H{h}')
    return encoded

def item_names(h):
    names = []
    for name in FEATURES:
        states = (['Deep', 'Medium', 'Shallow'] if name == 'SP500_Drawdown'
                  else ['Low', 'Medium', 'High'] if name in QUANTILE_FEATURES
                  else ['Negative', 'Zero', 'Positive'])
        names.extend(f'{name}={state}' for state in states)
    return names + [f'Gold_NonUp_H{h}', f'Gold_Up_H{h}']

def onehot(encoded, h):
    matrix = {}
    for item in item_names(h):
        field = item.split('=')[0] if '=' in item else 'Target_Item'
        matrix[item] = encoded[field].eq(item).to_numpy()
    return pd.DataFrame(matrix, index=encoded.index)

## 3. Apriori
Join và prune frequent itemsets; antecedent 1–3 item, consequent đúng một target. Support là tỷ lệ joint occurrence.

In [4]:
def apriori(matrix, min_support=0.02, max_size=4):
    """Breadth-first Apriori, with safe pruning of mutually exclusive states."""
    values = matrix.to_numpy(dtype=bool)
    names = matrix.columns.tolist()
    groups = [s.split('=')[0] if '=' in s else 'TARGET' for s in names]
    required = math.ceil(min_support * len(matrix) - 1e-12)
    previous = {(i,): int(values[:, i].sum()) for i in range(len(names))
                if values[:, i].sum() >= required}
    all_counts = dict(previous)
    for size in range(2, max_size + 1):
        keys = sorted(previous)
        candidates = set()
        
        for i, left in enumerate(keys):
            for right in keys[i+1:]:
                if left[:-1] != right[:-1]:
                    if right[:-1] > left[:-1]:
                        break
                    continue
                candidate = left + (right[-1],)
                if len({groups[j] for j in candidate}) != size:
                    continue
                if all(subset in previous for subset in combinations(candidate, size - 1)):
                    candidates.add(candidate)
        current = {}
        for candidate in sorted(candidates):
            count = int(values[:, candidate].all(axis=1).sum())
            if count >= required:
                current[candidate] = count
        all_counts.update(current)
        previous = current
        if not previous:
            break
    rows = [{'Items': ' & '.join(names[i] for i in key), 'Size': len(key),
             'Count': count, 'Support': count / len(matrix)} for key, count in all_counts.items()]
    return all_counts, pd.DataFrame(rows)

In [5]:
def generate_rules(counts, matrix, h):
    names = matrix.columns.tolist()
    target_indices = {names.index(f'Gold_Up_H{h}'), names.index(f'Gold_NonUp_H{h}')}
    rows = []
    for key, count in counts.items():
        targets = [i for i in key if i in target_indices]
        if len(targets) != 1 or not 2 <= len(key) <= 4:
            continue
        target = targets[0]
        antecedent = tuple(i for i in key if i != target)
        items = sorted(names[i] for i in antecedent)
        
        antecedent_count = counts[antecedent]
        consequent_count = counts[(target,)]
        confidence = count / antecedent_count
        base_rate = consequent_count / len(matrix)
        direction = 'Up' if names[target] == f'Gold_Up_H{h}' else 'NonUp'
        key_text = ' & '.join(items) + ' -> ' + direction
        rows.append({'Rule_ID': hashlib.sha256(key_text.encode()).hexdigest()[:20],
                     'Horizon': h, 'Antecedent': ' & '.join(items), 'Length': len(items),
                     'Consequent': names[target], 'Direction': direction,
                     'External': any(not item.startswith('Gold_') for item in items),
                     'Train_Rows': len(matrix), 'Train_Antecedent_Count': antecedent_count,
                     'Train_Joint_Count': count, 'Train_Support': count / len(matrix),
                     'Train_Confidence': confidence, 'Train_Lift': confidence / base_rate,
                     'Train_Base_Rate': base_rate})
    return pd.DataFrame(rows)

## 4. Đánh giá Validation và lưu ứng viên
Chỉ tính Train/Val. Test được thực hiện ở ARM-FINAL sau khi khóa cấu hình.

In [6]:
def measure_rules(rules, matrix, prefix):
    """Support is P(antecedent AND consequent), not antecedent coverage."""
    rows = []
    for rule in rules.itertuples(index=False):
        antecedent = matrix[rule.Antecedent.split(' & ')].all(axis=1).to_numpy()
        consequent = matrix[rule.Consequent].to_numpy()
        n_a = int(antecedent.sum())
        n_joint = int((antecedent & consequent).sum())
        base = float(consequent.mean())
        conf = n_joint / n_a if n_a else np.nan
        rows.append({f'{prefix}_Rows': len(matrix), f'{prefix}_Antecedent_Count': n_a,
                     f'{prefix}_Joint_Count': n_joint, f'{prefix}_Coverage': n_a / len(matrix),
                     f'{prefix}_Support': n_joint / len(matrix), f'{prefix}_Base_Rate': base,
                     f'{prefix}_Confidence': conf, f'{prefix}_Lift': conf / base if base else np.nan,
                     f'{prefix}_Leverage': n_joint / len(matrix) - n_a / len(matrix) * base})
    return pd.concat([rules.reset_index(drop=True), pd.DataFrame(rows)], axis=1)

In [7]:
def prepare_horizon(h):
    """Train + Validation only. No Test encoding, metrics or rule selection here."""
    out = folder(h)
    data = load_data(h)
    masks = split_masks(data)
    train, val = data.loc[masks['train']], data.loc[masks['val']]
    thresholds = fit_thresholds(train)
    matrices = {}
    data_out = DATA_OUT / f'ARM-H{h}'
    data_out.mkdir(parents=True, exist_ok=True)
    for split, frame in [('train', train), ('val', val)]:
        encoded = encode(frame, thresholds, h)
        encoded.to_csv(data_out / f'ARM_{split}.csv', index=False)
        matrices[split] = onehot(encoded, h)
        output = pd.concat([encoded[['Date', 'Target_Date']], matrices[split].astype('uint8')], axis=1)
        output.to_csv(data_out / f'ARM_{split}_onehot.csv', index=False)
    counts, frequent = apriori(matrices['train'])
    print('Số frequent itemset theo độ dài:'); display(frequent.groupby('Size').size().rename('Count'))
    rules = generate_rules(counts, matrices['train'], h)
    rules = measure_rules(rules, matrices['val'], 'Val')
    rules['Val_Stable'] = (rules.Val_Antecedent_Count >= MIN_VAL_COUNT) & (rules.Val_Lift > 1)
    rules.to_csv(out / f'H{h}_rules.csv', index=False)
    dropped = []
    for split, boundary, original in [
        ('train', '2021-01-01', data.Date < '2021-01-01'),
        ('val', '2024-01-01', data.Date.between('2021-01-01', '2023-12-31')),
    ]:
        part = data.loc[original & (data.Target_Date >= boundary), ['Date', 'Target_Date']].copy()
        part['Split'] = split
        part['Reason'] = 'Target date reaches next split'
        dropped.append(part)
    print('Phiên bị purge:'); display(pd.concat(dropped))
    preparation = {
        'horizon': h, 'train_rows': len(train), 'val_rows': len(val),
        'threshold_fit': 'Own purged Train; original unscaled features',
        'train_start': str(train.Date.min().date()), 'train_end': str(train.Date.max().date()),
        'val_start': str(val.Date.min().date()), 'val_end': str(val.Date.max().date()),
        'features': FEATURES, 'items': item_names(h), 'max_antecedent': 3,
        'source_sha256': digest(ROOT / f'data/modeling/horizons/h{h}/modeling_dataset.csv'),
        'trace_sha256': digest(ROOT / f'data/modeling/horizons/h{h}/target_trace.csv'),
        'candidate_sha256': digest(out / f'H{h}_rules.csv'),
    }
    state = {'thresholds': thresholds, 'preparation': preparation,
             'itemset_counts': frequent.groupby('Size').size().to_dict()}
    (data_out/'train_encoding.pkl').write_bytes(pickle.dumps(state))
    print('Ngưỡng chỉ fit trên Train:'); display(thresholds)
    print(f'H{h}: Train={len(train)}, Val={len(val)}, candidates={len(rules)}, '
          f'Val-stable={int(rules.Val_Stable.sum())}', flush=True)
    return rules

In [8]:
candidates = prepare_horizon(H)
print('Luật sinh trên Train, kèm chỉ số Validation để bước sau chọn cấu hình:'); display(candidates.head(10))

Số frequent itemset theo độ dài:


Size
1       28
2      356
3     2682
4    10990
Name: Count, dtype: int64

Phiên bị purge:


,Date,Target_Date,Split,Reason
4001,2020-12-03,2021-01-04,train,Target date reaches next split
4002,2020-12-04,2021-01-05,train,Target date reaches next split
4003,2020-12-07,2021-01-06,train,Target date reaches next split
4004,2020-12-08,2021-01-07,train,Target date reaches next split
4005,2020-12-09,2021-01-08,train,Target date reaches next split
4006,2020-12-10,2021-01-11,train,Target date reaches next split
4007,2020-12-11,2021-01-12,train,Target date reaches next split
4008,2020-12-14,2021-01-13,train,Target date reaches next split
4009,2020-12-15,2021-01-14,train,Target date reaches next split
4010,2020-12-16,2021-01-15,train,Target date reaches next split


Ngưỡng chỉ fit trên Train:


,Feature,q1,q2,Train_Rows,Interpolation
0,Gold_Intraday_Range,0.005864,0.012466,4001,linear
1,Gold_Volatility_20,0.007957,0.011064,4001,linear
2,Gold_RSI_14,46.602315,57.812843,4001,linear
3,SP500_Drawdown,-0.096311,-0.014672,4001,linear


H20: Train=4001, Val=734, candidates=4335, Val-stable=1803


Luật sinh trên Train, kèm chỉ số Validation để bước sau chọn cấu hình:


,Rule_ID,Horizon,Antecedent,Length,Consequent,Direction,External,Train_Rows,Train_Antecedent_Count,Train_Joint_Count,...,Val_Rows,Val_Antecedent_Count,Val_Joint_Count,Val_Coverage,Val_Support,Val_Base_Rate,Val_Confidence,Val_Lift,Val_Leverage,Val_Stable
0,7a6cfe55bff5987225d8,20,Gold_Daily_Return=Negative,1,Gold_NonUp_H20,NonUp,False,4001,1888,832,...,734,348,159,0.474114,0.216621,0.479564,0.456897,0.952733,-0.010747,False
1,f888f7be57eeab38ad5c,20,Gold_Daily_Return=Negative,1,Gold_Up_H20,Up,False,4001,1888,1056,...,734,348,189,0.474114,0.257493,0.520436,0.543103,1.043555,0.010747,True
2,0b9115788264bf8d8db3,20,Gold_Daily_Return=Positive,1,Gold_NonUp_H20,NonUp,False,4001,2090,942,...,734,382,191,0.520436,0.260218,0.479564,0.500000,1.042614,0.010636,True
3,bc7d4a3308d5d541afa7,20,Gold_Daily_Return=Positive,1,Gold_Up_H20,Up,False,4001,2090,1148,...,734,382,191,0.520436,0.260218,0.520436,0.500000,0.960733,-0.010636,False
4,167d43fa9b3a08972be0,20,Gold_Return_Lag_1=Negative,1,Gold_NonUp_H20,NonUp,False,4001,1888,826,...,734,347,163,0.472752,0.222071,0.479564,0.469741,0.979516,-0.004644,False
5,3c1474efaf28731b11c6,20,Gold_Return_Lag_1=Negative,1,Gold_Up_H20,Up,False,4001,1888,1062,...,734,347,184,0.472752,0.250681,0.520436,0.530259,1.018875,0.004644,True
6,dbc66adf91746c9df73e,20,Gold_Return_Lag_1=Positive,1,Gold_NonUp_H20,NonUp,False,4001,2090,952,...,734,383,187,0.521798,0.254768,0.479564,0.488251,1.018114,0.004533,True
7,13e8a548d778ca54ca60,20,Gold_Return_Lag_1=Positive,1,Gold_Up_H20,Up,False,4001,2090,1138,...,734,383,196,0.521798,0.267030,0.520436,0.511749,0.983309,-0.004533,False
8,bb4b831b7c6c2cbe36ef,20,Gold_Intraday_Range=Low,1,Gold_NonUp_H20,NonUp,False,4001,1334,578,...,734,15,8,0.020436,0.010899,0.479564,0.533333,1.112121,0.001099,False
9,bc5a1d16e2d90d5053fd,20,Gold_Intraday_Range=Low,1,Gold_Up_H20,Up,False,4001,1334,756,...,734,15,7,0.020436,0.009537,0.520436,0.466667,0.896684,-0.001099,False


## 5. Xét 27 cấu hình trên horizon này
Đây là thống kê riêng, chưa chọn cấu hình. ARM-FINAL cân bằng cả 8 nhóm horizon × hướng khi chọn cấu hình chung.

In [9]:
def grid_filter(rules, config):
    support, confidence, lift = config
    return (rules.Train_Support >= support) & (rules.Train_Confidence >= confidence) & (rules.Train_Lift >= lift)

In [10]:
rows = []
for config in GRID:
    passing = candidates.loc[grid_filter(candidates, config)]
    stable = passing.loc[passing.Val_Stable]
    rows.append({'min_support': config[0], 'min_confidence': config[1], 'min_lift': config[2],
                 'train_rules': len(passing), 'val_stable_rules': len(stable),
                 'val_up': int(stable.Direction.eq('Up').sum()),
                 'val_nonup': int(stable.Direction.eq('NonUp').sum())})
grid = pd.DataFrame(rows)
print('27 cấu hình — chỉ Train/Validation:')
display(grid)


27 cấu hình — chỉ Train/Validation:


,min_support,min_confidence,min_lift,train_rules,val_stable_rules,val_up,val_nonup
0,0.02,0.50,1.00,1307,536,453,83
1,0.02,0.50,1.05,844,335,252,83
2,0.02,0.50,1.10,533,184,101,83
3,0.02,0.55,1.00,1109,464,453,11
4,0.02,0.55,1.05,646,263,252,11
5,0.02,0.55,1.10,335,112,101,11
6,0.02,0.60,1.00,348,140,139,1
7,0.02,0.60,1.05,348,140,139,1
8,0.02,0.60,1.10,278,102,101,1
9,0.05,0.50,1.00,518,268,258,10
